In [0]:
from pyspark.sql import functions as F
S, G = "workspace.silver", "workspace.gold"

# dim_customer
(spark.table(f"{S}.customers")
   .select("customer_id","customer_unique_id","customer_city","customer_state")
   .write.format("delta").mode("overwrite").saveAsTable(f"{G}.dim_customer"))

# dim_product
(spark.table(f"{S}.products")
   .select("product_id","category","product_weight_g",
           "product_length_cm","product_height_cm","product_width_cm")
   .write.format("delta").mode("overwrite").saveAsTable(f"{G}.dim_product"))

# dim_seller
(spark.table(f"{S}.sellers")
   .select("seller_id","seller_city","seller_state")
   .write.format("delta").mode("overwrite").saveAsTable(f"{G}.dim_seller"))

# dim_date (generated calendar)
dates = spark.table(f"{S}.orders").agg(F.min("purchase_date").alias("a"),
                                       F.max("purchase_date").alias("b")).first()
cal = (spark.sql(f"SELECT explode(sequence(to_date('{dates.a}'), to_date('{dates.b}'), interval 1 day)) AS date")
    .withColumn("date_key", F.date_format("date","yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date")).withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date","MMM"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("day_of_week", F.date_format("date","E")))
cal.write.format("delta").mode("overwrite").saveAsTable(f"{G}.dim_date")

# fact_order_items (grain: one row per item in an order)
o = spark.table(f"{S}.orders")
i = spark.table(f"{S}.order_items")
fact = (i.join(o, "order_id")
  .select("order_id","order_item_id","customer_id","product_id","seller_id",
          F.date_format("purchase_date","yyyyMMdd").cast("int").alias("date_key"),
          "order_status","price","freight_value",
          (F.col("price")+F.col("freight_value")).alias("item_total"),
          F.datediff("order_delivered_customer_date","order_purchase_timestamp").alias("delivery_days"),
          F.datediff("order_delivered_customer_date","order_estimated_delivery_date").alias("days_vs_estimate"),
          (F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")).alias("is_late")))
fact.write.format("delta").mode("overwrite").saveAsTable(f"{G}.fact_order_items")

# fact_payments (grain: one row per payment per order)
(spark.table(f"{S}.order_payments")
   .select("order_id","payment_sequential","payment_type","payment_installments","payment_value")
   .write.format("delta").mode("overwrite").saveAsTable(f"{G}.fact_payments"))

In [0]:
from pyspark.sql import functions as F

class DQError(Exception):
    pass

def dq_check(table, pk=None, not_null=None, expected_cols=None,
             min_rows=1, fks=None):
    """
    pk: list of columns forming the primary key (uniqueness check)
    not_null: columns that must have no nulls
    expected_cols: columns that must exist (schema validation)
    fks: list of (col, ref_table, ref_col) for referential integrity
    """
    df = spark.table(table)
    results = []

    # row count
    n = df.count()
    results.append(("row_count>=%d" % min_rows, n >= min_rows, n))

    # schema validation
    if expected_cols:
        missing = [c for c in expected_cols if c not in df.columns]
        results.append(("schema_has_expected_cols", not missing, missing))

    # null checks
    for c in (not_null or []):
        nulls = df.filter(F.col(c).isNull()).count()
        results.append((f"not_null:{c}", nulls == 0, nulls))

    # uniqueness
    if pk:
        dups = n - df.select(*pk).distinct().count()
        results.append((f"unique:{'+'.join(pk)}", dups == 0, dups))

    # referential integrity
    for col, ref_table, ref_col in (fks or []):
        ref = spark.table(ref_table).select(F.col(ref_col).alias("_k")).distinct()
        orphans = (df.select(col).where(F.col(col).isNotNull()).distinct()
                     .join(ref, F.col(col) == F.col("_k"), "left_anti").count())
        results.append((f"fk:{col}->{ref_table}.{ref_col}", orphans == 0, orphans))

    failed = [r for r in results if not r[1]]
    print(f"--- DQ {table} ---")
    for name, ok, val in results:
        print(("PASS" if ok else "FAIL"), name, val)
    if failed:
        raise DQError(f"{table}: {len(failed)} check(s) failed")
    return True

In [0]:
dq_check("workspace.gold.fact_order_items", pk=["order_id","order_item_id"],
         not_null=["order_id","product_id","customer_id","seller_id","date_key"],
         fks=[("customer_id","workspace.gold.dim_customer","customer_id"),
              ("product_id","workspace.gold.dim_product","product_id"),
              ("seller_id","workspace.gold.dim_seller","seller_id"),
              ("date_key","workspace.gold.dim_date","date_key")])
dq_check("workspace.gold.dim_customer", pk=["customer_id"])
dq_check("workspace.gold.dim_product", pk=["product_id"])
dq_check("workspace.gold.dim_seller", pk=["seller_id"])
dq_check("workspace.gold.dim_date", pk=["date_key"])

--- DQ workspace.gold.fact_order_items ---
PASS row_count>=1 112650
PASS not_null:order_id 0
PASS not_null:product_id 0
PASS not_null:customer_id 0
PASS not_null:seller_id 0
PASS not_null:date_key 0
PASS unique:order_id+order_item_id 0
PASS fk:customer_id->workspace.gold.dim_customer.customer_id 0
PASS fk:product_id->workspace.gold.dim_product.product_id 0
PASS fk:seller_id->workspace.gold.dim_seller.seller_id 0
PASS fk:date_key->workspace.gold.dim_date.date_key 0
--- DQ workspace.gold.dim_customer ---
PASS row_count>=1 99441
PASS unique:customer_id 0
--- DQ workspace.gold.dim_product ---
PASS row_count>=1 32951
PASS unique:product_id 0
--- DQ workspace.gold.dim_seller ---
PASS row_count>=1 3095
PASS unique:seller_id 0
--- DQ workspace.gold.dim_date ---
PASS row_count>=1 774
PASS unique:date_key 0


True

In [0]:
#connectivity cell
from botocore.config import Config
import boto3

dbutils.widgets.text("aws_key", "")
dbutils.widgets.text("aws_secret", "")
BUCKET = "diksha-olist-lakehouse"

s3 = boto3.client(
    "s3",
    aws_access_key_id=dbutils.widgets.get("aws_key"),
    aws_secret_access_key=dbutils.widgets.get("aws_secret"),
    region_name="ap-southeast-2",
    config=Config(connect_timeout=10, retries={"max_attempts": 1}),
)
print(s3.list_objects_v2(Bucket=BUCKET, MaxKeys=1).get("KeyCount"))

1


In [0]:
import io

gold_tables = ["dim_customer", "dim_product", "dim_seller", "dim_date",
               "fact_order_items", "fact_payments"]

for t in gold_tables:
    pdf = spark.table(f"workspace.gold.{t}").toPandas()
    buf = io.StringIO()
    pdf.to_csv(buf, index=False)
    s3.put_object(Bucket=BUCKET, Key=f"gold/{t}/{t}.csv",
                  Body=buf.getvalue().encode("utf-8"))
    print(f"{t}: {len(pdf):,} rows -> s3://{BUCKET}/gold/{t}/{t}.csv")

dim_customer: 99,441 rows -> s3://diksha-olist-lakehouse/gold/dim_customer/dim_customer.csv
dim_product: 32,951 rows -> s3://diksha-olist-lakehouse/gold/dim_product/dim_product.csv
dim_seller: 3,095 rows -> s3://diksha-olist-lakehouse/gold/dim_seller/dim_seller.csv
dim_date: 774 rows -> s3://diksha-olist-lakehouse/gold/dim_date/dim_date.csv
fact_order_items: 112,650 rows -> s3://diksha-olist-lakehouse/gold/fact_order_items/fact_order_items.csv
fact_payments: 103,886 rows -> s3://diksha-olist-lakehouse/gold/fact_payments/fact_payments.csv


In [0]:
dbutils.widgets.removeAll()